# CardioM3Net — Full Training Pipeline
## Multimodal Meta-Learning Framework for Cardiovascular Disease Diagnosis

**Architecture:** ECG Encoder (ResNet1D + Attention) → Clinical Encoder (MLP) → Cross-Attention Fusion → Multi-Task Heads

**Training Pipeline:**
1. Self-Supervised Pretraining (SimCLR on ECG)
2. Multi-Task Supervised Training (Binary + Disease + Severity) with Domain Adaptation
3. MAML Meta-Learning (fast cross-dataset adaptation)
4. Federated Learning (privacy-preserving multi-hospital simulation)
5. Explainability (SHAP + Grad-CAM + Modality Attention)

**Datasets:** PTB-XL (21,837 ECGs) + UCI Heart Disease (clinical features)

---
### Setup: Add Datasets
1. Right sidebar → **Add Data** → search `m0hamedyousry/ptb-xl-a-large-publicly-available-ecg-dataset`
2. Right sidebar → **Add Data** → search `redwankarimsony/heart-disease-data`
3. Right sidebar → **Settings** → **Accelerator** → **GPU T4 x2**

In [ ]:
# STEP 1: Install dependencies
!pip install -q wfdb shap

In [ ]:
# STEP 2: Upload the cardiom3net/ folder and train_cardiom3net.py to /kaggle/working/
# If you uploaded as a zip:
# !unzip -q cardiom3net.zip -d /kaggle/working/

import os
print('Files in working dir:')
for f in os.listdir('/kaggle/working/'):
    print(f'  {f}')

In [ ]:
# STEP 3: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# STEP 4: Run FULL CardioM3Net training pipeline
# This runs ALL phases: SSL → Supervised → MAML → Federated → Explainability

!python /kaggle/working/train_cardiom3net.py \
    --epochs 30 \
    --ssl_epochs 20 \
    --batch_size 32 \
    --lr 0.001 \
    --maml_episodes 200 \
    --fed_rounds 5 \
    --fed_clients 3 \
    --output_dir /kaggle/working/cardiom3net_results

In [ ]:
# STEP 5: Quick run (skip SSL and MAML for faster testing)
# Uncomment below to run a faster version:

# !python /kaggle/working/train_cardiom3net.py \
#     --epochs 10 \
#     --batch_size 64 \
#     --skip_ssl \
#     --skip_maml \
#     --fed_rounds 3 \
#     --output_dir /kaggle/working/cardiom3net_results_quick

In [ ]:
# STEP 6: View results
import os
from IPython.display import Image, display

results_dir = '/kaggle/working/cardiom3net_results'

print('Generated files:')
for f in sorted(os.listdir(results_dir)):
    size = os.path.getsize(os.path.join(results_dir, f))
    print(f'  {f} ({size/1024:.0f} KB)')

# Display plots
for img in ['training_curves.png', 'confusion_matrices.png', 'roc_curves.png',
            'ecg_saliency.png', 'shap_clinical.png', 'modality_weights.png']:
    path = os.path.join(results_dir, img)
    if os.path.exists(path):
        print(f'\n--- {img} ---')
        display(Image(filename=path, width=800))

In [ ]:
# STEP 7: Load and inspect saved model
import torch

checkpoint = torch.load('/kaggle/working/cardiom3net_results/cardiom3net_central.pth',
                        map_location='cpu')

print('Model config:')
for k, v in checkpoint['config'].items():
    print(f'  {k}: {v}')

print('\nResults:')
for task, metrics in checkpoint['results'].items():
    print(f'  {task}:')
    for m, v in metrics.items():
        print(f'    {m}: {v:.4f}')